# Laboratorio 5

- Iris Ayala
- Anggie Quezada
- Jonathan Diaz

Territorio elegido: Suiza (código ISO CHE)

## Task 1.1

### a. Librerías

In [1]:
import os
import json
import zipfile
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import matplotlib.pyplot as plt
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from shapely.ops import unary_union
from dotenv import load_dotenv

print(gpd.__version__)
print(shapely.__version__)

1.2.0
2.1.2


### b. Polígonos administrativos de GADM

In [2]:
os.makedirs("datos", exist_ok=True)

def descargar(url, ruta):
    if not os.path.exists(ruta):
        r = requests.get(url)
        r.raise_for_status()
        with open(ruta, "wb") as f:
            f.write(r.content)

def cargar_gadm(nivel):
    nombre = "gadm41_CHE_" + str(nivel) + ".json"
    url = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/" + nombre + ".zip"
    ruta_zip = "datos/" + nombre + ".zip"
    descargar(url, ruta_zip)
    with zipfile.ZipFile(ruta_zip) as z:
        z.extractall("datos")
    return gpd.read_file("datos/" + nombre)

nivel1 = cargar_gadm(1)
nivel2 = cargar_gadm(2)

In [3]:
print(len(nivel1), nivel1.crs)
print(len(nivel2), nivel2.crs)
print(nivel1.crs.to_epsg() == 4326, nivel2.crs.to_epsg() == 4326)

26 EPSG:4326
169 EPSG:4326
True True


In [4]:
nivel1[["GID_1", "NAME_1"]].head()

,GID_1,NAME_1
0,CHE.1_1,Aargau
1,CHE.2_1,AppenzellAusserrhoden
2,CHE.3_1,AppenzellInnerrhoden
3,CHE.4_1,Basel-Landschaft
4,CHE.5_1,Basel-Stadt


El nivel 1 corresponde a los cantones y el nivel 2 a los distritos. Las dos capas vienen en WGS84 (EPSG:4326), o sea que las coordenadas están en grados de latitud y longitud.

### c. Instalaciones de salud de healthsites.io

In [ ]:
# la llave se lee del archivo .env (ver README)
load_dotenv()
API_KEY = os.getenv("HEALTHSITES_API_KEY")

ruta_salud = "datos/healthsites_CHE.geojson"

if not os.path.exists(ruta_salud):
    url = "https://healthsites.io/api/v3/facilities/"
    features = []
    pagina = 1
    while True:
        params = {"api-key": API_KEY, "country": "Switzerland", "page": pagina,
                  "output": "geojson", "flat-properties": "true"}
        r = requests.get(url, params=params)
        r.raise_for_status()
        datos = r.json()["features"]
        if len(datos) == 0:
            break
        features = features + datos
        pagina = pagina + 1
    with open(ruta_salud, "w") as f:
        json.dump({"type": "FeatureCollection", "features": features}, f)

salud_todo = gpd.read_file(ruta_salud)
salud_todo = salud_todo.set_crs(4326, allow_override=True)
len(salud_todo)

TypeError: string indices must be integers, not 'str'

In [ ]:
salud = gpd.sjoin(salud_todo, nivel1[["NAME_1", "geometry"]], predicate="intersects")
salud = salud.drop(columns="index_right")
salud = salud.drop_duplicates(subset="geometry")
salud["amenity"] = salud["amenity"].fillna("sin tipo")
len(salud)

In [ ]:
salud["amenity"].value_counts()

### d. Ráster de población de WorldPop

In [ ]:
url_pop = "https://data.worldpop.org/GIS/Population/Global_2000_2020_1km_UNadj/2020/CHE/che_ppp_2020_1km_Aggregated_UNadj.tif"
ruta_pop = "datos/che_ppp_2020_1km.tif"
descargar(url_pop, ruta_pop)

with rasterio.open(ruta_pop) as src:
    banda = src.read(1, masked=True)
    lat_centro = (src.bounds.top + src.bounds.bottom) / 2
    res_y = src.res[1] * 111320
    res_x = src.res[0] * 111320 * np.cos(np.radians(lat_centro))
    print(src.crs)
    print(src.res)
    print(res_x, res_y)
    print(src.height, src.width)
    print(banda.max())
    print(banda.max() / (res_x * res_y / 1e6))
    print(banda.sum())

El ráster viene en EPSG:4326 con píxeles de 30 segundos de arco (0.00833 grados). Como está en grados, la resolución en metros se calcula aproximando: un grado de latitud son unos 111.32 km, y en longitud se multiplica por el coseno de la latitud. Por eso el píxel no es cuadrado en metros: a la latitud de Suiza mide unos 930 m de norte a sur y unos 630 m de este a oeste, o sea cerca de 0.59 km².

Cada píxel guarda el número estimado de personas en esa celda. El valor máximo del ráster es entonces el máximo de personas en una celda, y para tenerlo como densidad en habitantes por km² se divide dentro del área del píxel. La suma de todos los píxeles da la población total del país.

## Task 1.2

### a. Reproyección

Se eligió EPSG:2056 (CH1903+ / LV95), que es el sistema de coordenadas oficial de Suiza.

En EPSG:4326 las coordenadas están en grados y un grado no mide lo mismo en todas partes. A la latitud de Suiza un grado de longitud mide unos 76 km y uno de latitud unos 111 km, entonces un buffer de "10 km" hecho en grados saldría deformado. Se necesita una proyección en metros.

LV95 usa una proyección de Mercator oblicua conforme (conserva formas y ángulos). El cilindro toca el elipsoide a lo largo de un círculo que pasa por Berna (46.95°N, 7.44°E), donde el factor de escala es k0 = 1. La distorsión de distancias crece con la distancia norte-sur a esa línea, aproximadamente como:

k ≈ 1 + y² / (2R²)

donde y es la distancia perpendicular a la línea de contacto y R el radio de la Tierra.

Esta proyección minimiza la distorsión para Suiza porque fue diseñada para el país: Suiza es alargada de este a oeste (unos 350 km) y angosta de norte a sur (unos 220 km), y la línea de contacto va justo en dirección este-oeste pasando por el centro. Entonces ningún punto del país queda lejos de la línea sin distorsión.

El peor caso es el extremo sur (Ticino, 45.82°N), que está a unos 126 km de la línea de Berna. Con R = 6378 km:

k ≈ 1 + 126² / (2 × 6378²) ≈ 1.0002

O sea, el error máximo de distancia es de alrededor de 0.02%. En un buffer de 20 km eso son unos 4 m, que no es nada comparado con el tamaño de un píxel de población.

También se pudo usar UTM zona 32N (EPSG:32632), pero su meridiano central está en 9°E y tiene factor de escala 0.9996, así que el error llega a cerca de 0.04%. Funciona, pero LV95 es mejor porque está centrada en el país.

In [ ]:
crs_m = "EPSG:2056"

nivel1_m = nivel1.to_crs(crs_m)
nivel2_m = nivel2.to_crs(crs_m)
salud_m = salud.to_crs(crs_m)
salud_m["geometry"] = salud_m.centroid

ruta_pop_m = "datos/che_ppp_2020_1km_m.tif"

with rasterio.open(ruta_pop) as src:
    transform, ancho, alto = calculate_default_transform(src.crs, crs_m, src.width, src.height, *src.bounds)
    meta = src.meta.copy()
    meta.update(crs=crs_m, transform=transform, width=ancho, height=alto)
    with rasterio.open(ruta_pop_m, "w", **meta) as dst:
        reproject(source=rasterio.band(src, 1),
                  destination=rasterio.band(dst, 1),
                  src_transform=src.transform,
                  src_crs=src.crs,
                  dst_transform=transform,
                  dst_crs=crs_m,
                  resampling=Resampling.sum)

pop = rasterio.open(ruta_pop_m)
print(pop.crs, pop.res)
print(pop.read(1, masked=True).sum())

Para el ráster se usó el remuestreo por suma porque los píxeles son conteos de personas. Si se usara vecino más cercano o bilineal la población total cambiaría al reproyectar.

### b. Mapa base

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
nivel2_m.plot(ax=ax, color="lightgray", edgecolor="white", linewidth=0.3)
salud_m.plot(ax=ax, column="amenity", markersize=8, legend=True, cmap="tab10")
nivel1_m.boundary.plot(ax=ax, color="black", linewidth=0.8)

minx, miny, maxx, maxy = nivel1_m.total_bounds
x0 = minx + 20000
y0 = miny + 20000
ax.plot([x0, x0 + 50000], [y0, y0], color="black", linewidth=3)
ax.text(x0 + 25000, y0 + 5000, "50 km", ha="center")

ax.annotate("N", xy=(0.95, 0.97), xytext=(0.95, 0.87), xycoords="axes fraction",
            ha="center", fontsize=14, arrowprops=dict(facecolor="black", width=3, headwidth=10))

ax.set_title("Instalaciones de salud en Suiza")
ax.set_xlabel("Este (m)")
ax.set_ylabel("Norte (m)")
plt.show()

### c. Estadísticas por cantón

In [ ]:
def poblacion(geom):
    datos, t = mask(pop, [geom], crop=True, filled=False)
    return float(datos.sum())

def distancia_vecino(puntos):
    n = len(puntos)
    if n < 2:
        return np.nan
    x = puntos.geometry.x.values
    y = puntos.geometry.y.values
    d = np.sqrt((x[:, None] - x[None, :])**2 + (y[:, None] - y[None, :])**2)
    np.fill_diagonal(d, np.inf)
    return d.min(axis=1).mean() / 1000

filas = []
for i, fila in nivel1_m.iterrows():
    puntos = salud_m[salud_m["NAME_1"] == fila["NAME_1"]]
    pob = poblacion(fila.geometry)
    filas.append({"canton": fila["NAME_1"],
                  "instalaciones": len(puntos),
                  "area_km2": fila.geometry.area / 1e6,
                  "poblacion": pob,
                  "inst_por_100k": len(puntos) / pob * 100000,
                  "dist_vecino_km": distancia_vecino(puntos)})

tabla = pd.DataFrame(filas).sort_values("inst_por_100k")
tabla.round(2)

In [ ]:
tabla.head(3)

## Task 1.3

### a. Buffers de cobertura

In [ ]:
radios = [5, 10, 20]
cobertura = {}
for r in radios:
    buffers = salud_m.buffer(r * 1000)
    cobertura[r] = unary_union(list(buffers))

territorio = unary_union(list(nivel1_m.geometry))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, r in zip(axes, radios):
    nivel1_m.plot(ax=ax, color="lightgray", edgecolor="black", linewidth=0.5)
    gpd.GeoSeries([cobertura[r].intersection(territorio)], crs=crs_m).plot(ax=ax, color="tab:blue", alpha=0.5)
    ax.set_title("Buffer de " + str(r) + " km")
    ax.set_axis_off()
plt.show()

### b. Cobertura poblacional

In [ ]:
pob_total = poblacion(territorio)

filas = []
for r in radios:
    cubierta = poblacion(cobertura[r].intersection(territorio))
    filas.append({"radio_km": r,
                  "poblacion_cubierta": cubierta,
                  "poblacion_total": pob_total,
                  "porcentaje": cubierta / pob_total * 100})

tabla_cob = pd.DataFrame(filas)
tabla_cob.round(2)

### c. Mapa de cobertura a 10 km

In [ ]:
cob10 = cobertura[10]

def clasificar(geom):
    if cob10.covers(geom):
        return "Completamente cubierto"
    elif cob10.intersects(geom):
        return "Parcialmente cubierto"
    else:
        return "No cubierto"

nivel2_m["cobertura"] = nivel2_m.geometry.apply(clasificar)
nivel2_m["cobertura"].value_counts()

In [ ]:
cmap = plt.get_cmap("RdYlBu")
colores = {"No cubierto": cmap(0.1), "Parcialmente cubierto": cmap(0.5), "Completamente cubierto": cmap(0.9)}

fig, ax = plt.subplots(figsize=(10, 10))
for categoria in colores:
    parte = nivel2_m[nivel2_m["cobertura"] == categoria]
    if len(parte) > 0:
        parte.plot(ax=ax, color=colores[categoria], edgecolor="gray", linewidth=0.3)
    ax.scatter([], [], color=colores[categoria], marker="s", s=80, label=categoria)

nivel1_m.boundary.plot(ax=ax, color="black", linewidth=0.8)
salud_m.plot(ax=ax, color="black", markersize=3, label="Instalación de salud")
ax.legend(loc="lower right")
ax.set_title("Cobertura de servicios de salud con radio de 10 km")
ax.set_axis_off()
plt.show()